In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Base

In [2]:
base_path = Path('data/processed/base_final.parquet')
if not base_path.exists():
    base_path = Path('base_final.parquet')

base = pd.read_parquet(base_path)

In [3]:
base.head()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,load_mean_3h,load_std_3h,load_mean_6h,load_std_6h,load_mean_24h,load_std_24h,ramp_1h,target_load_1h,target_load_6h,target_load_24h
0,NE,2022-01-08 00:00:00,11926.333,2022,0.520000,974.148148,974.203704,973.448148,47.800000,24.492593,...,12420.102000,134.929127,12237.736833,221.810606,11584.597667,583.661108,-244.169,11635.094,9793.342,11660.956
1,NE,2022-01-08 01:00:00,11635.094,2022,0.253333,973.319643,973.403571,972.975000,1.060000,24.185714,...,12233.397333,292.610669,12222.878833,241.764288,11583.075958,582.678831,-338.512,11402.214,9865.790,11290.670
2,NE,2022-01-08 02:00:00,11402.214,2022,0.238298,974.063793,974.348276,973.979310,16.883333,24.013793,...,11942.090667,315.171078,12139.194667,343.039821,11584.575750,582.768185,-291.239,11147.334,10122.028,11049.698
3,NE,2022-01-08 03:00:00,11147.334,2022,0.486957,973.198214,973.580357,973.164286,615.487500,23.726786,...,11654.547000,262.600449,12037.324500,459.007934,11585.088542,582.594832,-232.880,10927.972,10205.706,10924.900
4,NE,2022-01-08 04:00:00,10927.972,2022,0.195652,973.167273,973.587273,973.147273,426.871429,23.521818,...,11394.880667,243.962677,11814.139000,518.641087,11581.656042,585.017287,-254.880,10275.910,10321.871,10726.672


In [4]:
base.tail()

,subsystem_code,timestamp,load_mw,source_year,precipitation_mm,pressure_station_hpa,pressure_max_hpa,pressure_min_hpa,global_radiation_kj_m2,temperature_c,...,load_mean_3h,load_std_3h,load_mean_6h,load_std_6h,load_mean_24h,load_std_24h,ramp_1h,target_load_1h,target_load_6h,target_load_24h
40699,NE,2026-08-30 19:00:00,14732.475,2026,0.021212,975.022973,975.251351,974.924324,1297.912329,31.492105,...,13593.778000,1245.447100,12371.095833,1555.554662,12931.276833,1580.179317,1089.576,14748.239,13415.721,15309.979
40700,NE,2026-08-30 20:00:00,14748.239,2026,0.209091,975.313514,975.333784,974.989189,589.445946,30.151316,...,14404.422333,614.416698,12972.719167,1670.015240,12899.577292,1533.490668,-52.709,14907.176,13135.991,15255.274
40701,NE,2026-08-30 21:00:00,14907.176,2026,0.080000,975.983562,976.000000,975.584932,107.198630,28.260000,...,14755.299333,27.054499,13589.470833,1490.593328,12877.769458,1501.682676,15.764,14852.579,12924.672,15686.557
40702,NE,2026-08-30 22:00:00,14852.579,2026,0.120635,976.952113,976.957746,976.387324,33.144444,26.986301,...,14795.963333,96.634977,14194.870667,1028.477317,12862.479292,1477.901832,158.937,14383.492,12865.152,15660.982
40703,NE,2026-08-30 23:00:00,14383.492,2026,0.022581,978.195652,978.198551,977.484058,48.388235,25.819718,...,14835.998000,80.755431,14620.210167,457.699677,12853.685417,1464.804863,-54.597,13805.914,12934.367,15149.486


In [5]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40704 entries, 0 to 40703
Data columns (total 51 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   subsystem_code          40704 non-null  string        
 1   timestamp               40704 non-null  datetime64[us]
 2   load_mw                 40704 non-null  float64       
 3   source_year             40704 non-null  int64         
 4   precipitation_mm        40704 non-null  float64       
 5   pressure_station_hpa    40704 non-null  float64       
 6   pressure_max_hpa        40704 non-null  float64       
 7   pressure_min_hpa        40704 non-null  float64       
 8   global_radiation_kj_m2  40704 non-null  float64       
 9   temperature_c           40704 non-null  float64       
 10  dew_point_c             40704 non-null  float64       
 11  temperature_max_c       40704 non-null  float64       
 12  temperature_min_c       40704 non-null  float6

# Baseline

## 1h

In [8]:
# t+1 = t
# prever proximo horario = carga atual
base['baseline_persistence'] = base['load_mw']

# t+1 = (t+1)-24
# prever proximo horario = carga do mesmo horario do dia anterior
base['baseline_yesterday'] = base['load_mw'].shift(23)

# t+1 = (t+1)-168
# prever proximo horario = carga do mesmo horario da semana anterior
base['baseline_last_week'] = base['load_mw'].shift(167)

In [11]:
baseline_df = base[[
    'target_load_1h',
    'baseline_persistence',
    'baseline_yesterday',
    'baseline_last_week'
]].dropna()

In [14]:
# MAE

mae_persistence = mean_absolute_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_persistence']
)

mae_yesterday = mean_absolute_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_yesterday']
)

mae_last_week = mean_absolute_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_last_week']
)

# RMSE

rmse_persistence = np.sqrt(
    mean_squared_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_persistence']
    )
)

rmse_yesterday = np.sqrt(
    mean_squared_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_yesterday']
    )
)

rmse_last_week = np.sqrt(
    mean_squared_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_last_week']
    )
)

# MAPE

mape_persistence = mean_absolute_percentage_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_persistence']
)

mape_yesterday = mean_absolute_percentage_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_yesterday']
)

mape_last_week = mean_absolute_percentage_error(
    baseline_df['target_load_1h'],
    baseline_df['baseline_last_week']
)

In [15]:
print(f'Persistence      | MAE: {mae_persistence:.2f} | RMSE: {rmse_persistence:.2f} | MAPE: {mape_persistence:.2%}')
print(f'Mesmo horário ontem | MAE: {mae_yesterday:.2f} | RMSE: {rmse_yesterday:.2f} | MAPE: {mape_yesterday:.2%}')
print(f'Semana anterior  | MAE: {mae_last_week:.2f} | RMSE: {rmse_last_week:.2f} | MAPE: {mape_last_week:.2%}')

Persistence      | MAE: 294.02 | RMSE: 373.36 | MAPE: 2.35%
Mesmo horário ontem | MAE: 595.95 | RMSE: 910.10 | MAPE: 4.93%
Semana anterior  | MAE: 465.35 | RMSE: 651.57 | MAPE: 3.75%


## 6h

In [16]:
# Persistence: assume que daqui a 6h será igual a agora
base['baseline_persistence_6h'] = base['load_mw']

# Mesmo horário do dia anterior
# 24 - 6 = 18
base['baseline_yesterday_6h'] = base['load_mw'].shift(18)

# Mesmo horário da semana anterior
# 168 - 6 = 162
base['baseline_last_week_6h'] = base['load_mw'].shift(162)

In [17]:
baseline6_df = base[[
    'target_load_6h',
    'baseline_persistence_6h',
    'baseline_yesterday_6h',
    'baseline_last_week_6h'
]].dropna()

In [18]:
# MAE

mae_persistence6 = mean_absolute_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_persistence_6h']
)

mae_yesterday6 = mean_absolute_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_yesterday_6h']
)

mae_last_week6 = mean_absolute_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_last_week_6h']
)

# RMSE

rmse_persistence6 = np.sqrt(
    mean_squared_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_persistence_6h']
    )
)

rmse_yesterday6 = np.sqrt(
    mean_squared_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_yesterday_6h']
    )
)

rmse_last_week6 = np.sqrt(
    mean_squared_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_last_week_6h']
    )
)

# MAPE

mape_persistence6 = mean_absolute_percentage_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_persistence_6h']
)

mape_yesterday6 = mean_absolute_percentage_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_yesterday_6h']
)

mape_last_week6 = mean_absolute_percentage_error(
    baseline6_df['target_load_6h'],
    baseline6_df['baseline_last_week_6h']
)

print(f'Persistence      | MAE: {mae_persistence6:.2f} | RMSE: {rmse_persistence6:.2f} | MAPE: {mape_persistence6:.2%}')
print(f'Mesmo horário ontem | MAE: {mae_yesterday6:.2f} | RMSE: {rmse_yesterday6:.2f} | MAPE: {mape_yesterday6:.2%}')
print(f'Semana anterior  | MAE: {mae_last_week6:.2f} | RMSE: {rmse_last_week6:.2f} | MAPE: {mape_last_week6:.2%}')

Persistence      | MAE: 1123.66 | RMSE: 1358.56 | MAPE: 9.11%
Mesmo horário ontem | MAE: 595.90 | RMSE: 910.05 | MAPE: 4.93%
Semana anterior  | MAE: 465.32 | RMSE: 651.53 | MAPE: 3.75%


## 24

In [19]:
base['baseline_persistence_24h'] = base['load_mw']

# 24 - 24 = 0
base['baseline_yesterday_24h'] = base['load_mw']

# 168 - 24 = 144
base['baseline_last_week_24h'] = base['load_mw'].shift(144)

In [20]:
baseline24_df = base[[
    'target_load_24h',
    'baseline_persistence_24h',
    'baseline_yesterday_24h',
    'baseline_last_week_24h'
]].dropna()

In [21]:
# MAE

mae_persistence24 = mean_absolute_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_persistence_24h']
)

mae_yesterday24 = mean_absolute_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_yesterday_24h']
)

mae_last_week24 = mean_absolute_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_last_week_24h']
)

# RMSE

rmse_persistence24 = np.sqrt(
    mean_squared_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_persistence_24h']
    )
)

rmse_yesterday24 = np.sqrt(
    mean_squared_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_yesterday_24h']
    )
)

rmse_last_week24 = np.sqrt(
    mean_squared_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_last_week_24h']
    )
)

# MAPE

mape_persistence24 = mean_absolute_percentage_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_persistence_24h']
)

mape_yesterday24 = mean_absolute_percentage_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_yesterday_24h']
)

mape_last_week24 = mean_absolute_percentage_error(
    baseline24_df['target_load_24h'],
    baseline24_df['baseline_last_week_24h']
)

print(f'Persistence      | MAE: {mae_persistence24:.2f} | RMSE: {rmse_persistence24:.2f} | MAPE: {mape_persistence24:.2%}')
print(f'Mesmo horário ontem | MAE: {mae_yesterday24:.2f} | RMSE: {rmse_yesterday24:.2f} | MAPE: {mape_yesterday24:.2%}')
print(f'Semana anterior  | MAE: {mae_last_week24:.2f} | RMSE: {rmse_last_week24:.2f} | MAPE: {mape_last_week24:.2%}')

Persistence      | MAE: 596.66 | RMSE: 911.67 | MAPE: 4.93%
Mesmo horário ontem | MAE: 596.66 | RMSE: 911.67 | MAPE: 4.93%
Semana anterior  | MAE: 465.32 | RMSE: 651.51 | MAPE: 3.75%
